In [1]:
import sys
import time
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from utils import set_seed, count_parameters
from data_loader import read_raw_data
from graph_construction import get_all_atc_l2_codes
from models import SimpleMLPModel
from train_eval import train, model_test
import config


def train_test(data_train, data_test, data_all, args):
    data_train = np.array(data_train)
    data_test = np.array(data_test)
    data_all = np.array(data_all)

    (fingerprints, mol_graphs, adr_graph, adr_node2idx, index_to_adrecs, 
     atc_graph, atc_node2idx, index_to_atc) = read_raw_data(data_train, data_test, args)
    
    atc_l2_list = get_all_atc_l2_codes(index_to_atc)
    print(f"\n=== ATC L2 Statistics ===")
    print(f"Total unique L2 codes: {len(atc_l2_list)}")
    print(f"Sample L2 codes: {atc_l2_list[:10]}")
    
    all_classes = atc_l2_list + [None]
    
    trainset = torch.utils.data.TensorDataset(
        torch.LongTensor(data_train[:, 0]),
        torch.LongTensor(data_train[:, 1]),
        torch.LongTensor(data_train[:, 2]),
    )
    testset = torch.utils.data.TensorDataset(
        torch.LongTensor(data_test[:, 0]),
        torch.LongTensor(data_test[:, 1]),
        torch.LongTensor(data_test[:, 2]),
    )
    allset = torch.utils.data.TensorDataset(
        torch.LongTensor(data_all[:, 0]),
        torch.LongTensor(data_all[:, 1]),
        torch.LongTensor(data_all[:, 2]),
    )
    
    device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")
    
    pin_memory = True if device.type == "cuda" else False
    num_workers = 8 if device.type == "cuda" else 0
    
    _train = torch.utils.data.DataLoader(trainset, batch_size=args.batch_size, shuffle=True, 
                                        pin_memory=pin_memory, num_workers=num_workers)
    _test = torch.utils.data.DataLoader(testset, batch_size=args.test_batch_size, shuffle=False, 
                                       pin_memory=pin_memory, num_workers=num_workers)
    _all = torch.utils.data.DataLoader(allset, batch_size=args.test_batch_size, shuffle=False,
                                      pin_memory=pin_memory, num_workers=num_workers)
    
    model = SimpleMLPModel(
        fp_dim=config.MORGAN_FP_DIM, 
        adr_feature_dim=64, 
        atc_feature_dim=64, 
        embed_dim=args.embed_dim, 
        atc_l2_list=atc_l2_list, 
        mol_graphs=mol_graphs,
        dropout=args.droprate
    ).to(device)
    
    total_params = count_parameters(model)
    print(f"Total number of trainable parameters: {total_params}")
    
    model.set_graph(adr_graph, adr_node2idx, index_to_adrecs, 
                    atc_graph, atc_node2idx, index_to_atc)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=config.LR_SCHEDULER_FACTOR, patience=config.LR_SCHEDULER_PATIENCE
    )
    
    best_test_f1 = 0
    best_test_roc_auc = 0
    best_model_path = 'best_model.pth'
    best_pred_path = 'best_test_predictions.csv'
    best_all_pred_path = 'best_all_predictions.csv'
    endure_count = 0
    start = time.time()

    data_test_original = pd.DataFrame(data_test, columns=['drug_index', 'adr_index', 'label'])
    data_all_original = pd.DataFrame(data_all, columns=['drug_index', 'adr_index', 'label'])
    
    for epoch in range(1, args.epochs + 1):
        train_loss, train_f1, train_precision, train_recall, train_roc_auc, train_pr_auc, train_accuracy, train_lb_loss = train(
            fingerprints, adr_graph, adr_node2idx, index_to_adrecs, atc_graph, atc_node2idx, index_to_atc, 
            model, _train, optimizer, criterion, device, atc_l2_list, all_classes)
        
        test_loss, test_f1, test_precision, test_recall, test_roc_auc, test_pr_auc, test_accuracy, test_preds = model_test(
            fingerprints, adr_graph, adr_node2idx, index_to_adrecs, atc_graph, atc_node2idx, index_to_atc, 
            model, _test, device, atc_l2_list, all_classes)

        all_loss, all_f1, all_precision, all_recall, all_roc_auc, all_pr_auc, all_accuracy, all_preds = model_test(
            fingerprints, adr_graph, adr_node2idx, index_to_adrecs, atc_graph, atc_node2idx, index_to_atc,
            model, _all, device, atc_l2_list, all_classes)
        
        scheduler.step(test_loss)
        time_cost = time.time() - start
        
        print(f"Time: {time_cost:.2f}s | Epoch: {epoch} | LR: {optimizer.param_groups[0]['lr']:.6f}")
        print(f"[Train] Loss: {train_loss:.4f} | LB Loss: {train_lb_loss:.4f} | F1: {train_f1:.4f} | ROC-AUC: {train_roc_auc:.4f} | "
              f"PR-AUC: {train_pr_auc:.4f} | Precision: {train_precision:.4f} | Recall: {train_recall:.4f} | "
              f"Accuracy: {train_accuracy:.4f}")
        print(f"[Test]  Loss: {test_loss:.4f} | F1: {test_f1:.4f} | ROC-AUC: {test_roc_auc:.4f} | "
              f"PR-AUC: {test_pr_auc:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f} | "
              f"Accuracy: {test_accuracy:.4f}")
        print(f"[All]   Loss: {all_loss:.4f} | F1: {all_f1:.4f} | ROC-AUC: {all_roc_auc:.4f} | "
              f"PR-AUC: {all_pr_auc:.4f} | Precision: {all_precision:.4f} | Recall: {all_recall:.4f} | "
              f"Accuracy: {all_accuracy:.4f}")
        print("-" * 100)
        
        if test_roc_auc > best_test_roc_auc:
            best_test_roc_auc = test_roc_auc
            best_test_f1 = test_f1
            endure_count = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'test_f1': test_f1,
                'test_roc_auc': test_roc_auc,
            }, best_model_path)
            pred_df = data_test_original.copy()
            pred_df['pred_score'] = test_preds
            pred_df.to_csv(best_pred_path, index=False)
            all_pred_df = data_all_original.copy()
            all_pred_df['pred_score'] = all_preds
            all_pred_df.to_csv(best_all_pred_path, index=False)
            print(f"✓ Model saved at epoch {epoch} with ROC-AUC: {test_roc_auc:.4f}, F1: {test_f1:.4f}")
            print(f"✓ Predictions saved to {best_pred_path}")
            print(f"✓ All predictions saved to {best_all_pred_path}\n")
        else:
            endure_count += 1
        
        if endure_count > config.EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break
    
    checkpoint = torch.load(best_model_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    with torch.no_grad():
        adr_embeddings = model.adr_rgcn(model.adr_graph.to(device))
        atc_embeddings = model.atc_rgcn(model.atc_graph.to(device))
    
    pt_embeddings = {}
    for idx, adrecs_ids in index_to_adrecs.items():
        for adrecs_id in adrecs_ids:
            pt_idx = adr_node2idx.get(adrecs_id, None)
            if pt_idx is not None:
                pt_embeddings[f"{idx}_{adrecs_id}"] = adr_embeddings[pt_idx].cpu().numpy()
    
    atc_level5_embeddings = {}
    for idx, atc_codes in index_to_atc.items():
        for atc_code in atc_codes:
            atc_idx = atc_node2idx.get(atc_code, None)
            if atc_idx is not None:
                atc_level5_embeddings[(idx, atc_code)] = atc_embeddings[atc_idx].cpu().numpy()
    
    embedding_df = pd.DataFrame.from_dict(pt_embeddings, orient="index")
    embedding_df.to_csv("adr_embeddings_learned.csv")
    
    atc_embedding_df = pd.DataFrame.from_dict(atc_level5_embeddings, orient="index")
    atc_embedding_df.to_csv("atc_embeddings_learned.csv")
    
    return best_test_roc_auc


def main():
    parser = argparse.ArgumentParser(description='Drug-ADR Prediction with Dual MoE and GNN')
    parser.add_argument('--epochs', type=int, default=config.EPOCHS, help='number of epochs to train')
    parser.add_argument('--lr', type=float, default=config.LEARNING_RATE, help='learning rate')
    parser.add_argument('--embed_dim', type=int, default=config.EMBED_DIM, help='embedding dimension')
    parser.add_argument('--weight_decay', type=float, default=config.WEIGHT_DECAY, help='weight decay')
    parser.add_argument('--droprate', type=float, default=config.DROPOUT_RATE, help='dropout rate')
    parser.add_argument('--batch_size', type=int, default=config.BATCH_SIZE, help='input batch size for training')
    parser.add_argument('--test_batch_size', type=int, default=config.TEST_BATCH_SIZE, help='input batch size for testing')
    parser.add_argument('--rawpath', type=str, default='/data/', help='rawpath')
    parser.add_argument('--seed', type=int, default=config.SEED, help='random seed for reproducibility')
    args, _ = parser.parse_known_args()
    
    set_seed(args.seed)
    
    print('-------------------- Hyperparams --------------------')
    print(f'random seed: {args.seed}')
    print(f'weight decay: {args.weight_decay}')
    print(f'dropout rate: {args.droprate}')
    print(f'learning rate: {args.lr}')
    print(f'dimension of embedding: {args.embed_dim}')
    print('-------------------- Feature Dimensions --------------------')
    print(f'Morgan fingerprint: {config.MORGAN_FP_DIM}')
    print(f'Morgan reduced: {config.MORGAN_REDUCED_DIM}')
    print(f'GNN output: {config.GNN_OUTPUT_DIM}')
    print(f'Fused drug feature: {config.FUSED_DRUG_DIM} (Morgan {config.MORGAN_REDUCED_DIM} + GNN {config.GNN_OUTPUT_DIM})')
    print(f'Atom feature dimension: 46')
    
    try:
        data_train = pd.read_csv('train.csv')
        data_test = pd.read_csv('test.csv')
        data_all = pd.read_csv('all.csv')
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Required file not found: {e}")
    
    data_train = np.array(data_train).astype(int)
    data_test = np.array(data_test).astype(int)
    data_all = np.array(data_all).astype(int)
    
    association_auc = train_test(data_train, data_test, data_all, args)
    
    print(f"\n{'='*100}")
    print(f"Final Best Test ROC-AUC: {association_auc:.5f}")
    print(f"{'='*100}")
    sys.stdout.flush()


if __name__ == "__main__":
    main()

-------------------- Hyperparams --------------------
random seed: 32
weight decay: 1e-05
dropout rate: 0.5
learning rate: 0.0002
dimension of embedding: 64
-------------------- Feature Dimensions --------------------
Morgan fingerprint: 1024
Morgan reduced: 256
GNN output: 32
Fused drug feature: 288 (Morgan 256 + GNN 32)
Atom feature dimension: 46


[14:10:12] WARNING: not removing hydrogen atom without neighbors
[14:10:12] WARNING: not removing hydrogen atom without neighbors
[14:10:12] WARNING: not removing hydrogen atom without neighbors
[14:10:12] WARNING: not removing hydrogen atom without neighbors
[14:10:12] WARNING: not removing hydrogen atom without neighbors
[14:10:12] WARNING: not removing hydrogen atom without neighbors
[14:10:12] WARNING: not removing hydrogen atom without neighbors


Building molecular graphs...


[14:10:13] WARNING: not removing hydrogen atom without neighbors
[14:10:13] WARNING: not removing hydrogen atom without neighbors
[14:10:13] WARNING: not removing hydrogen atom without neighbors
[14:10:13] WARNING: not removing hydrogen atom without neighbors
[14:10:14] WARNING: not removing hydrogen atom without neighbors
[14:10:14] WARNING: not removing hydrogen atom without neighbors
[14:10:14] WARNING: not removing hydrogen atom without neighbors


Built 1775 molecular graphs

=== ADR Graph (PT Co-occurrence) Jaccard Statistics ===
Total co-occurrence pairs calculated: 1630802
Jaccard Statistics (all pairs):
  Mean: 0.041827
  Std:  0.027987
  Min:  0.001447
  Max:  0.360000
  Median: 0.034884

Dynamic threshold (percentile 75): 0.055556

After filtering (>= 0.055556):
  Number of edges: 413595
  Percentage retained: 25.36%
  Mean Jaccard: 0.080595
  Std Jaccard:  0.023917
  Min/Max Jaccard: 0.055556 / 0.360000

=== ATC Graph (L5 Co-occurrence) Jaccard Statistics ===
Total co-occurrence pairs calculated: 2716690
Jaccard Statistics (all pairs):
  Mean: 0.060475
  Std:  0.053682
  Min:  0.000829
  Max:  1.000000
  Median: 0.050847

Dynamic threshold (percentile 75): 0.082474

After filtering (>= 0.082474):
  Number of edges: 679693
  Percentage retained: 25.02%
  Mean Jaccard: 0.121483
  Std Jaccard:  0.071862
  Min/Max Jaccard: 0.082474 / 1.000000

=== ATC L2 Statistics ===
Total unique L2 codes: 87
Sample L2 codes: ['A01', 'A02',

KeyboardInterrupt: 

In [3]:
import pandas as pd
from sklearn.metrics import f1_score, classification_report

# ── 1. 读取两个文件 ───────────────────────────────────────────────────────────
pred_df = pd.read_csv("best_test_predictions.csv")
atc_df  = pd.read_csv("result_with_atc_traintest.csv")

# ── 2. 构建 drug_index → ATC 码 列表 的映射 ──────────────────────────────────
# atc_df 按行顺序对应 index 0, 1, 2, ...
atc_df = atc_df.reset_index(drop=True)          # 确保 index 从 0 连续
atc_df["drug_index"] = atc_df.index             # 新增一列：药物 index

# ── 3. 筛选含有 ATC 第二层级 "D03" 的药物 ────────────────────────────────────
TARGET_LEVEL2 = "D03"

def has_target_level2(atc_str, target):
    """检查分号分隔的 ATC 串中是否有某个第二层级 code（前3位）"""
    for code in str(atc_str).split(";"):
        if code.strip()[:3] == target:
            return True
    return False

mask = atc_df["ATC_CODE"].apply(lambda x: has_target_level2(x, TARGET_LEVEL2))
target_drug_indices = set(atc_df.loc[mask, "drug_index"].tolist())

print(f"含 {TARGET_LEVEL2} 的药物数量：{len(target_drug_indices)}")
print(f"对应 drug_index：{sorted(target_drug_indices)}")

# ── 4. 从预测结果中筛出这些药物的样本 ─────────────────────────────────────────
filtered_df = pred_df[pred_df["drug_index"].isin(target_drug_indices)].copy()
print(f"\n筛出样本数：{len(filtered_df)}")
print(filtered_df)

# ── 5. 将 pred_score 二值化（阈值 0.5）并计算 F1 ─────────────────────────────
THRESHOLD = 0.5
filtered_df["pred_label"] = (filtered_df["pred_score"] >= THRESHOLD).astype(int)

y_true = filtered_df["label"]
y_pred = filtered_df["pred_label"]

# 宏平均 / 二分类 F1
f1_binary = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
f1_macro  = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"\n── F1 结果（阈值={THRESHOLD}）──────────────────────────")
print(f"  Binary F1 (pos=1)  : {f1_binary:.4f}")
print(f"  Macro  F1          : {f1_macro:.4f}")
print("\n── 详细分类报告 ─────────────────────────────────────")
print(classification_report(y_true, y_pred, zero_division=0))

含 D03 的药物数量：1
对应 drug_index：[405]

筛出样本数：23
       drug_index  adr_index  label  pred_score
47099         405       1058      0    0.002428
47347         405        831      0    0.000269
47749         405        359      0    0.003106
47990         405       1196      0    0.037701
48591         405       2044      0    0.001519
49822         405        475      0    0.000990
50066         405       1126      0    0.000393
50456         405       1892      0    0.001420
50886         405        204      0    0.001510
51843         405       1175      0    0.001090
51915         405       2017      0    0.007601
52483         405        612      0    0.000880
52985         405       1071      0    0.000391
54386         405        657      0    0.004160
54948         405        605      0    0.004875
55312         405       1769      0    0.001997
55463         405        494      0    0.632245
56083         405       1667      0    0.017732
56097         405       1298      0    0.007